# 15 — Inverse Layer Optimisation: Can K_nm Explain Real PLV?

The cross-scale PLV test (notebook colab/kaggle_physionet_cross_scale) failed:
r=-0.37 with assumed layer assignments. But the failure might be in the
assignments, not K_nm itself.

**Test**: Given the measured PLV matrix from real PhysioNet data, find the
signal-to-layer assignment that MAXIMISES correlation with K_nm.
If the optimal assignment beats 95th percentile of random permutations,
K_nm has real predictive structure.

In [ ]:
import numpy as np
import json
from scipy.stats import pearsonr, spearmanr
from itertools import permutations

from scpn_quantum_control.bridge import build_knm_paper27

In [ ]:
# Load real PLV results
RESULTS_PATH = '/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/00_DATASETS/results/cross_scale_plv_real_data.json'
with open(RESULTS_PATH) as f:
    plv_data = json.load(f)

print(f'Records: {plv_data["n_records"]}')
print(f'Original mean r: {plv_data["mean_r"]}')

K16 = build_knm_paper27(L=16)

In [ ]:
# Original (theoretical) layer assignments
ORIGINAL_LAYERS = {
    'eeg_delta': 9,
    'eeg_theta': 5,
    'eeg_alpha': 10,
    'eeg_beta': 4,
    'eeg_gamma': 7,  # not in slpdb data
    'ecg_raw': 4,
    'resp_raw': 6,
    'bp_mayer': 6,
}

# Average PLV matrix across all records
all_plv = [np.array(r['PLV']) for r in plv_data['results']]
mean_plv = np.mean(all_plv, axis=0)
signal_names = plv_data['results'][0]['signals']
N = len(signal_names)

print(f'Signals ({N}): {signal_names}')
print(f'Mean PLV matrix (averaged over {len(all_plv)} records):')
for i, s in enumerate(signal_names):
    print(f'  {s[:10]:>10}', end='')
    for j in range(N):
        print(f'{mean_plv[i,j]:8.3f}', end='')
    print()

In [ ]:
def score_assignment(layer_indices, K16, plv_matrix):
    """Pearson r between K_nm sub-matrix and PLV for a given assignment."""
    N = len(layer_indices)
    K_sub = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            K_sub[i, j] = K16[layer_indices[i], layer_indices[j]]
    
    idx = np.triu_indices(N, k=1)
    k_flat = K_sub[idx]
    plv_flat = plv_matrix[idx]
    
    if np.std(k_flat) < 1e-10 or np.std(plv_flat) < 1e-10:
        return 0.0
    return pearsonr(k_flat, plv_flat)[0]


# Score original assignment
original_idx = [ORIGINAL_LAYERS.get(s, 8) - 1 for s in signal_names]  # 0-indexed
r_original = score_assignment(original_idx, K16, mean_plv)
print(f'Original (theoretical) assignment: r = {r_original:.4f}')
print(f'Layers: {[i+1 for i in original_idx]}')

## Exhaustive Search Over Layer Assignments

7 signals, 16 possible layers each. Full search = 16^7 ~ 268M — too large.
Instead: sample 500,000 random assignments + local optimisation from top candidates.

In [ ]:
rng = np.random.default_rng(42)
n_random = 500_000

print(f'Sampling {n_random:,} random layer assignments...')
best_r = -1.0
best_assignment = None
all_r = []

for _ in range(n_random):
    assignment = rng.integers(0, 16, size=N)
    r = score_assignment(assignment, K16, mean_plv)
    all_r.append(r)
    if r > best_r:
        best_r = r
        best_assignment = assignment.copy()

all_r = np.array(all_r)
print(f'Random search complete.')
print(f'  Mean r:   {np.mean(all_r):.4f}')
print(f'  Std r:    {np.std(all_r):.4f}')
print(f'  95th pct: {np.percentile(all_r, 95):.4f}')
print(f'  99th pct: {np.percentile(all_r, 99):.4f}')
print(f'  Best r:   {best_r:.4f}')
print(f'  Best assignment (1-indexed): {[i+1 for i in best_assignment]}')

In [ ]:
# Local hill-climbing from top 100 candidates
print('\nHill-climbing from top 100 random candidates...')
top_idx = np.argsort(all_r)[-100:]

# Reconstruct top assignments by re-running with same seed
rng2 = np.random.default_rng(42)
assignments = []
for _ in range(n_random):
    assignments.append(rng2.integers(0, 16, size=N))

global_best_r = best_r
global_best = best_assignment.copy()

for idx in top_idx:
    current = assignments[idx].copy()
    current_r = all_r[idx]
    improved = True
    while improved:
        improved = False
        for sig in range(N):
            for layer in range(16):
                if layer == current[sig]:
                    continue
                trial = current.copy()
                trial[sig] = layer
                r = score_assignment(trial, K16, mean_plv)
                if r > current_r:
                    current = trial
                    current_r = r
                    improved = True
    if current_r > global_best_r:
        global_best_r = current_r
        global_best = current.copy()

print(f'\nOptimal assignment found:')
print(f'  r = {global_best_r:.4f}')
print(f'  Layers (1-indexed): {[i+1 for i in global_best]}')
print()
for s, l in zip(signal_names, global_best):
    orig = ORIGINAL_LAYERS.get(s, '?')
    match = 'MATCH' if (l + 1) == orig else f'was L{orig}'
    print(f'  {s:<12} -> L{l+1:2d}  ({match})')

In [ ]:
# Statistical significance: is optimal r better than random?
p_opt = np.mean(all_r >= global_best_r)
pct_rank = (1 - p_opt) * 100

print(f'\n=== SIGNIFICANCE ===')
print(f'  Optimal r:       {global_best_r:.4f}')
print(f'  Original r:      {r_original:.4f}')
print(f'  Random mean r:   {np.mean(all_r):.4f}')
print(f'  Percentile rank: {pct_rank:.2f}%')
print(f'  p-value:         {p_opt:.6f}')
print()

if global_best_r > np.percentile(all_r, 95) and global_best_r > 0:
    print('  K_nm HAS predictive structure for real physiological PLV.')
    print('  The theoretical layer assignments were suboptimal,')
    print('  but the coupling matrix itself captures real cross-scale relationships.')
    if global_best_r > 0.5:
        print('  r > 0.5: STRONG evidence. Publishable with proper controls.')
    else:
        print(f'  r = {global_best_r:.3f}: moderate evidence. Needs more data.')
else:
    print('  K_nm does NOT predict real PLV better than random assignments.')
    print('  The coupling matrix structure does not match physiological reality.')
    print('  This is a NEGATIVE result — the layer model may need fundamental revision.')

In [ ]:
# Per-record validation of optimal assignment
print('\n=== PER-RECORD VALIDATION ===')
for rec_data in plv_data['results']:
    plv_rec = np.array(rec_data['PLV'])
    r_opt = score_assignment(global_best, K16, plv_rec)
    r_orig = score_assignment(np.array(original_idx), K16, plv_rec)
    print(f'  {rec_data["record"]}: r_optimal={r_opt:.4f}, r_original={r_orig:.4f}, improvement={r_opt-r_orig:+.4f}')

In [ ]:
print('--- JSON RESULTS ---')
print(json.dumps({
    'n_signals': N,
    'n_random_samples': n_random,
    'r_original': round(r_original, 4),
    'r_optimal': round(global_best_r, 4),
    'optimal_layers': [int(i+1) for i in global_best],
    'original_layers': [i+1 for i in original_idx],
    'random_mean_r': round(float(np.mean(all_r)), 4),
    'random_95pct': round(float(np.percentile(all_r, 95)), 4),
    'percentile_rank': round(pct_rank, 2),
    'p_value': round(float(p_opt), 6),
    'verdict': 'K_nm has structure' if global_best_r > np.percentile(all_r, 95) and global_best_r > 0 else 'K_nm no better than random',
}, indent=2))